In [ ]:
"""
This code will add a bunch of presets to the geoguesser presets. These include heritage sites, famous places, and more.
"""

import math
import json

# ── Core geometry ─────────────────────────────────────────────────────────────

def circle_polygon(lat: float, lng: float,
                   radius_km: float, n_points: int = 48) -> list:
    """
    Return a closed polygon (list of [lat, lng]) approximating a circle
    centred at (lat, lng) with the given radius in kilometres.

    Uses the spherical-Earth destination formula so the circle is accurate
    everywhere — not just near the equator.
    """
    R = 6371.0
    lat_r = math.radians(lat)
    lng_r = math.radians(lng)
    d_r   = radius_km / R          # angular distance in radians

    points = []
    for i in range(n_points):
        bearing = math.radians(360.0 * i / n_points)
        dest_lat = math.asin(
            math.sin(lat_r) * math.cos(d_r) +
            math.cos(lat_r) * math.sin(d_r) * math.cos(bearing)
        )
        dest_lng = lng_r + math.atan2(
            math.sin(bearing) * math.sin(d_r) * math.cos(lat_r),
            math.cos(d_r) - math.sin(lat_r) * math.sin(dest_lat)
        )
        points.append([
            round(math.degrees(dest_lat), 6),
            round(math.degrees(dest_lng), 6),
        ])

    points.append(points[0])   # close the ring
    return points


def build_preset_polygons(centers: list) -> list:
    """
    centers: list of (lat, lng, radius_km)
    Returns: list of polygon rings  (the 'polygons' field for geo_preset_create)
    """
    return [circle_polygon(lat, lng, r) for lat, lng, r in centers]


# ── Preset definitions ────────────────────────────────────────────────────────
# Format per entry: (lat, lng, radius_km)
#   radius_km controls how tightly the circle hugs the site.
#   Small iconic landmark  →  1–3 km
#   City centre            →  8–15 km
#   National park / region →  30–120 km

PRESETS = [

    # ── World Heritage & Ancient Wonders ─────────────────────────────────────
    {
        "title": "Ancient Wonders of the World",
        "username": "system",
        "centers": [
            (29.9792,  31.1342,   3.0),   # Great Pyramid of Giza
            (37.9395,  27.3408,   2.0),   # Temple of Artemis, Ephesus
            (37.0380,  27.4241,   1.5),   # Mausoleum at Halicarnassus
            (36.4513,  28.2278,   1.5),   # Colossus of Rhodes (approximate)
            (27.9559,  34.3234,   2.0),   # Lighthouse of Alexandria (Pharos)
            (31.5000,  35.4500,   2.0),   # Hanging Gardens of Babylon (approximate Iraq)
            (37.6387,  22.7276,   2.0),   # Statue of Zeus at Olympia
        ],
    },
    {
        "title": "UNESCO World Heritage: Natural Wonders",
        "username": "system",
        "centers": [
            (-25.3444, -131.0235,  15.0),  # Galápagos Islands
            ( 27.9881,  86.9250,   20.0),  # Sagarmatha / Everest NP
            (-13.1631, -72.5450,    3.0),  # Machu Picchu
            (-19.8834, -43.9036,   12.0),  # Iguazú Falls area
            ( 46.8800,   8.0000,   30.0),  # Swiss Alps / Jungfrau
            (-34.4100,  19.3400,   25.0),  # Cape Floral Region, S. Africa
            (  1.3700, 103.8200,    8.0),  # Singapore Botanic Gardens
        ],
    },
    {
        "title": "UNESCO World Heritage: Ancient Cities",
        "username": "system",
        "centers": [
            (30.3285,  35.4444,   5.0),   # Petra, Jordan
            (37.8719,  34.6847,   4.0),   # Göbekli Tepe / Çatalhöyük area
            (17.3754,  78.4760,   5.0),   # Golconda / Qutb Shahi Tombs, Hyderabad
            (29.9753,  31.1376,   4.0),   # Memphis & Saqqara, Egypt
            (15.5518,  32.5324,   4.0),   # Ancient Meroe, Sudan
            (35.7215,  51.3347,   8.0),   # Persepolis / Tehran area
            (41.0082,  28.9784,   6.0),   # Constantinople / Istanbul old city
        ],
    },

    # ── European Icons ────────────────────────────────────────────────────────
    {
        "title": "Iconic European Landmarks",
        "username": "system",
        "centers": [
            (48.8584,   2.2945,   2.0),   # Eiffel Tower, Paris
            (51.5007,  -0.1246,   2.0),   # Big Ben, London
            (41.9009,  12.4833,   2.0),   # Colosseum, Rome
            (47.5580,   7.5880,   2.0),   # Basel / Rhine falls area
            (50.0755,  14.4378,   5.0),   # Prague Old Town
            (52.5163,  13.3777,   2.0),   # Brandenburg Gate, Berlin
            (40.4168,  -3.7038,   5.0),   # Madrid centre
            (37.9715,  23.7267,   3.0),   # Acropolis, Athens
        ],
    },
    {
        "title": "Medieval Castles of Europe",
        "username": "system",
        "centers": [
            (47.5578,  10.7498,   2.0),   # Neuschwanstein, Germany
            (53.7632,  -0.3373,   2.5),   # Conisburgh Castle
            (49.3231,  -0.6764,   2.0),   # Château de Caen, France
            (55.9485,  -3.1999,   2.0),   # Edinburgh Castle
            (51.4080,  -3.5795,   2.0),   # Caerphilly Castle, Wales
            (48.2500,   7.4833,   2.0),   # Château du Haut-Koenigsbourg
            (43.7696,   7.6850,   2.0),   # Monaco / Château Grimaldi
            (50.9326,  14.0567,   2.0),   # Oybin Castle, Germany
        ],
    },
    {
        "title": "Scenic Mediterranean Coast",
        "username": "system",
        "centers": [
            (43.7315,   7.4197,   8.0),   # Côte d'Azur, France
            (40.5460,  14.2345,   8.0),   # Amalfi Coast, Italy
            (35.5018,  24.0116,  12.0),   # Crete coastline
            (42.6508,   2.9030,  10.0),   # Costa Brava, Spain
            (36.8969,  11.1176,  10.0),   # Sidi Bou Said, Tunisia
            (35.8833,  14.4167,   8.0),   # Malta coastline
            (38.4192,  26.1230,   8.0),   # Izmir / Cesme coast, Turkey
        ],
    },
    {
        "title": "Scandinavian Fjords & Nature",
        "username": "system",
        "centers": [
            (61.2100,   7.1000,  25.0),   # Sognefjord, Norway
            (60.3913,   5.3221,  10.0),   # Bergen, Norway
            (63.4305,  10.3951,  10.0),   # Trondheim area
            (68.1000,  14.1000,  20.0),   # Lofoten Islands
            (65.0121,  25.4651,  10.0),   # Oulu / Finnish coast
            (59.3293,  18.0686,   8.0),   # Stockholm archipelago
            (64.9631, -19.0208,  30.0),   # Icelandic highlands
        ],
    },

    # ── Asia Pacific ─────────────────────────────────────────────────────────
    {
        "title": "Iconic Landmarks of Asia",
        "username": "system",
        "centers": [
            (27.1751,  78.0421,   3.0),   # Taj Mahal, Agra
            (31.1328, 121.4991,   8.0),   # The Bund, Shanghai
            (35.6586, 139.7454,   2.0),   # Tokyo Tower
            (22.2796, 114.1624,   5.0),   # Hong Kong harbour
            (13.4125, 103.8670,  10.0),   # Angkor Wat, Cambodia
            ( 3.1570, 101.7120,   3.0),   # Petronas Towers, KL
            (37.5760, 126.9770,   8.0),   # Seoul Gyeongbokgung area
        ],
    },
    {
        "title": "Japan: Temples, Cities & Nature",
        "username": "system",
        "centers": [
            (34.9671, 135.7727,   8.0),   # Kyoto (Fushimi Inari area)
            (35.6762, 139.6503,   8.0),   # Tokyo (Shinjuku)
            (34.6937, 135.5023,   8.0),   # Osaka castle area
            (43.0642, 141.3469,   8.0),   # Sapporo, Hokkaido
            (35.3606, 138.7274,   8.0),   # Mt Fuji
            (34.3955, 132.4596,   5.0),   # Hiroshima Peace Memorial
            (26.2124, 127.6792,   8.0),   # Naha, Okinawa
        ],
    },
    {
        "title": "Southeast Asian Temples & Coastlines",
        "username": "system",
        "centers": [
            (13.4125, 103.8670,  10.0),   # Angkor Wat, Cambodia
            ( 8.5069,  99.8406,  15.0),   # Koh Samui, Thailand
            ( 5.4141, 100.3288,   8.0),   # Penang heritage area
            (-8.5069, 115.2625,  12.0),   # Bali, Ubud / Kuta
            ( 2.0469, 102.5714,   5.0),   # Malacca historic city
            (14.0583, 108.2772,  20.0),   # Hội An / Đà Nẵng, Vietnam
            (10.0452, 105.7469,  10.0),   # Mekong Delta, Vietnam
        ],
    },
    {
        "title": "Great Wall of China Sections",
        "username": "system",
        "centers": [
            (40.4319, 116.5704,   6.0),   # Mutianyu section
            (40.3594, 116.0154,   6.0),   # Juyongguan pass
            (40.6769, 117.2334,   6.0),   # Simatai section
            (39.5600, 114.5500,  10.0),   # Jinshanling / Shanxi area
            (40.7714, 119.7506,   8.0),   # Laolongtou (Old Dragon Head) at sea
        ],
    },
    {
        "title": "Australian Outback & Coasts",
        "username": "system",
        "centers": [
            (-25.3444, 131.0369,  10.0),  # Uluru (Ayers Rock)
            (-33.8688, 151.2093,  10.0),  # Sydney Harbour
            (-27.1167, 153.0333,   8.0),  # Sunshine Coast
            (-17.9644, 122.2304,  15.0),  # Broome / Cable Beach
            (-16.9186, 145.7781,  10.0),  # Cairns / Great Barrier Reef gateway
            (-43.5321, 172.6362,  10.0),  # Christchurch, NZ
            (-44.9950, 168.6562,  20.0),  # Milford Sound, NZ
        ],
    },

    # ── The Americas ──────────────────────────────────────────────────────────
    {
        "title": "US National Parks",
        "username": "system",
        "centers": [
            (36.4864, -112.0900,  30.0),  # Grand Canyon
            (44.5978, -110.5642,  40.0),  # Yellowstone
            (37.7459, -119.5332,  20.0),  # Yosemite Valley
            (38.6800,  -109.5000, 30.0),  # Arches NP, Utah
            (64.7511, -141.0000,  80.0),  # Denali NP, Alaska
            (25.6866,  -80.4473,  20.0),  # Everglades
            (47.8021, -123.6044,  25.0),  # Olympic NP
        ],
    },
    {
        "title": "American Southwest Desert",
        "username": "system",
        "centers": [
            (36.9983, -110.0988,  15.0),  # Monument Valley
            (38.5733, -109.5498,  20.0),  # Canyonlands
            (37.2970, -112.9408,  20.0),  # Zion NP
            (37.6283, -112.1677,  15.0),  # Bryce Canyon
            (33.4484, -112.0740,  10.0),  # Phoenix desert fringe
            (32.2540, -110.9742,   8.0),  # Tucson / Saguaro NP
            (35.4676, -105.9072,   8.0),  # Santa Fe, New Mexico
        ],
    },
    {
        "title": "Latin American Wonders",
        "username": "system",
        "centers": [
            (-13.1631,  -72.5450,   4.0),  # Machu Picchu
            (-22.9068,  -43.1729,  10.0),  # Rio de Janeiro / Sugarloaf
            (-16.5000,  -68.1500,  10.0),  # La Paz / Lake Titicaca area
            (-25.6947,  -54.4365,   8.0),  # Iguazú Falls
            (-33.4489,  -70.6693,  10.0),  # Santiago, Chile
            ( 19.4326,  -99.1332,  10.0),  # Mexico City / Teotihuacán
            ( 20.6843,  -88.5678,   5.0),  # Chichén Itzá, Mexico
        ],
    },
    {
        "title": "Caribbean Islands",
        "username": "system",
        "centers": [
            (18.4861,  -69.9312,  15.0),  # Santo Domingo, Dominican Republic
            (18.1096,  -77.2975,  20.0),  # Jamaica (Kingston / Blue Mountains)
            (25.0480,  -77.3554,  15.0),  # Nassau, Bahamas
            (13.1132,  -59.5988,  12.0),  # Bridgetown, Barbados
            (17.1274,  -61.8468,  10.0),  # Antigua
            (10.6918,  -61.2225,  10.0),  # Port of Spain, Trinidad
            (16.2650,  -61.5510,  12.0),  # Guadeloupe
        ],
    },

    # ── Africa & Middle East ──────────────────────────────────────────────────
    {
        "title": "African Safari Regions",
        "username": "system",
        "centers": [
            (-2.3333,  34.8333,  40.0),   # Serengeti, Tanzania
            (-3.2440,  37.8251,  15.0),   # Kilimanjaro
            ( 0.3476,  37.5880,  30.0),   # Tsavo / Amboseli, Kenya
            (-19.9545,  23.5887, 40.0),   # Okavango Delta, Botswana
            (-24.0210,  31.4870, 30.0),   # Kruger National Park, S. Africa
            (-17.8600,  25.8572, 20.0),   # Victoria Falls, Zambia/Zimbabwe
            ( 9.1450,  40.4897,  40.0),   # Ethiopian Highlands / Rift Valley
        ],
    },
    {
        "title": "Middle East & North Africa",
        "username": "system",
        "centers": [
            (29.9792,  31.1342,   5.0),   # Giza pyramids, Egypt
            (30.3285,  35.4444,   5.0),   # Petra, Jordan
            (25.2048,  55.2708,   8.0),   # Dubai skyline
            (24.4539,  54.3773,   8.0),   # Abu Dhabi
            (33.5138,  36.2765,   8.0),   # Damascus old city
            (33.8886,  35.4955,   6.0),   # Beirut
            (24.6877,  46.7219,   8.0),   # Riyadh, Saudi Arabia
        ],
    },
    {
        "title": "Nile Valley & Ancient Egypt",
        "username": "system",
        "centers": [
            (29.9792,  31.1342,   4.0),   # Giza / Great Pyramid
            (25.7289,  32.6102,   5.0),   # Luxor / Karnak Temple
            (24.0889,  32.8998,   5.0),   # Aswan / Abu Simbel region
            (30.0626,  31.2497,   8.0),   # Cairo historic centre
            (27.2446,  31.0722,   4.0),   # Abydos / Dendera
        ],
    },

    # ── Thematic / Specialty ──────────────────────────────────────────────────
    {
        "title": "Volcanic Islands & Calderas",
        "username": "system",
        "centers": [
            (28.2916,  -16.6291,  20.0),  # Tenerife / Teide, Canary Islands
            (19.4069, -155.2834,  25.0),  # Hawaii Big Island / Kilauea
            (64.9631,  -19.0208,  40.0),  # Iceland volcanic plateau
            (-8.5069,  115.2625,  15.0),  # Bali / Mount Agung
            (37.7290,  15.0050,   8.0),   # Mount Etna, Sicily
            (-14.2350, -170.7094, 15.0),  # American Samoa
            ( 0.6684,  127.3868,  20.0),  # North Maluku, Indonesia
        ],
    },
    {
        "title": "World's Greatest Deserts",
        "username": "system",
        "centers": [
            (23.4162,  25.6628,  80.0),   # Sahara, Libya / Egypt border
            (27.0,     -13.0,    80.0),   # Western Sahara
            (23.0,     57.0,     50.0),   # Rub' al Khali (Empty Quarter), Oman
            (19.0,     56.5,     50.0),   # Wahiba Sands, Oman
            (36.9983, -110.0988, 30.0),   # Navajo desert, USA
            (-23.698,  133.882,  60.0),   # Simpson Desert, Australia
            (-26.0,    24.0,     80.0),   # Kalahari, Botswana/Namibia
        ],
    },
    {
        "title": "Tropical Rainforests",
        "username": "system",
        "centers": [
            (-3.4653,  -62.2159,  80.0),  # Amazon Basin, Brazil
            ( 0.4162,   9.4673,  40.0),   # Congo Rainforest, Gabon
            ( 4.2,    117.5,     40.0),   # Borneo interior, Malaysia
            ( 5.5,    -1.5,      30.0),   # Ghana / Ivory Coast forest
            (-5.5,    144.0,     40.0),   # Papua New Guinea highlands
            ( 7.0,    -73.5,     30.0),   # Colombian Amazon
            (15.0,    100.0,     25.0),   # Mekong / Cardamom forest, SE Asia
        ],
    },
    {
        "title": "High-Altitude Landscapes",
        "username": "system",
        "centers": [
            (27.9881,  86.9250,  25.0),   # Nepal Himalayas / Everest
            (30.7333,  79.0667,  30.0),   # Uttarakhand / Garhwal Himalayas
            (-16.5000, -68.1500, 20.0),   # Bolivian Altiplano
            (31.5540,  75.0649, 25.0),    # Dharamsala / Spiti Valley
            (63.4696,  20.3010, 15.0),    # Highlands of Iceland (Askja)
            (45.8326,   6.8652,  20.0),   # Mont Blanc / French Alps
            (-33.6534, -70.0110, 30.0),   # Andes, Chile/Argentina border
        ],
    },
    {
        "title": "World-Famous Beaches",
        "username": "system",
        "centers": [
            (-22.9714, -43.1823,   5.0),  # Copacabana, Rio de Janeiro
            ( 7.8804,  98.2976,   10.0),  # Phuket / Patong, Thailand
            (21.2793, -157.8319,   8.0),  # Waikiki, Hawaii
            (43.7315,   7.4197,   8.0),   # Nice / Côte d'Azur
            (-8.6500, 115.1500,   8.0),   # Kuta Beach, Bali
            (25.7617,  -80.1918,   8.0),  # Miami Beach
            (36.3520,  25.4615,   8.0),   # Santorini beaches
            (37.8719,  34.6847,   8.0),   # Antalya coast, Turkey
        ],
    },
    {
        "title": "World Capitals: Architecture",
        "username": "system",
        "centers": [
            (48.8566,   2.3522,   8.0),   # Paris
            (51.5074,  -0.1278,   8.0),   # London
            (55.7558,  37.6173,   8.0),   # Moscow
            (39.9042, 116.4074,   8.0),   # Beijing
            (35.6762, 139.6503,   8.0),   # Tokyo
            (28.6139,  77.2090,   8.0),   # New Delhi
            (-15.7942, -47.8825,  8.0),   # Brasília
            ( 4.3517,  18.5582,   8.0),   # Bangui — unusual capital
        ],
    },
    {
        "title": "Island Nations",
        "username": "system",
        "centers": [
            ( 3.2028,  73.2207,  20.0),   # Maldives (Male atoll)
            (-4.6191,  55.4513,  15.0),   # Seychelles (Mahé)
            (13.1130, -59.5988,  12.0),   # Barbados
            (-21.1333,-175.2000, 20.0),   # Tonga
            (-13.9170, -171.9549,15.0),   # Samoa
            ( 7.3697,  134.4706, 20.0),   # Palau
            (-8.9000,  160.0500, 20.0),   # Solomon Islands
        ],
    },
    {
        "title": "UNESCO: Historic City Centres",
        "username": "system",
        "centers": [
            (50.0755,  14.4378,   5.0),   # Prague
            (47.4979,  19.0402,   6.0),   # Budapest
            (59.4370,  24.7536,   6.0),   # Tallinn
            (54.6872,  25.2797,   5.0),   # Vilnius
            (56.9460,  24.1059,   5.0),   # Riga
            (44.8025,  20.4651,   6.0),   # Belgrade
            (43.8563,  18.4131,   5.0),   # Sarajevo
            (41.9981,  21.4254,   5.0),   # Skopje old bazaar
        ],
    },
    {
        "title": "Famous Rivers of the World",
        "username": "system",
        "centers": [
            (29.9792,  31.1342,  10.0),   # Nile delta / Giza
            (-3.4653,  -60.018,  30.0),   # Amazon, Manaus
            (30.5928, 114.3055,  10.0),   # Yangtze, Wuhan
            (48.8566,   2.3522,   5.0),   # Seine, Paris
            (51.5074,  -0.1278,   5.0),   # Thames, London
            (55.7558,  37.6173,   5.0),   # Moscow River
            (17.3850,  78.4867,  10.0),   # Godavari / Ganges delta
            ( 0.0,    -67.0,     30.0),   # Orinoco, Venezuela
        ],
    },
    # ── Roman Empire Trail ────────────────────────────────────────────────────
    {
        "title": "Roman Empire: Greatest Sites",
        "username": "system",
        "centers": [
            (41.8902,  12.4922,   3.0),   # Colosseum, Rome
            (37.6284,  14.3311,   3.0),   # Villa Romana del Casale, Sicily
            (36.8528,  10.3233,   5.0),   # Carthage ruins, Tunisia
            (32.9069,  13.1875,   5.0),   # Leptis Magna, Libya
            (43.9467,   3.6667,  10.0),   # Pont du Gard / Nîmes, France
            (41.6486,  -4.7234,   5.0),   # Segovia aqueduct, Spain
            (51.1788,  -1.8262,   3.0),   # Bath (Aquae Sulis), England
            (44.5500,  33.5167,   5.0),   # Chersonesus, Crimea
        ],
    },
    {
        "title": "Hellenistic & Ancient Greek World",
        "username": "system",
        "centers": [
            (37.9715,  23.7267,   3.0),   # Acropolis, Athens
            (37.6387,  22.7276,   3.0),   # Ancient Olympia
            (40.4920,  22.9887,   5.0),   # Vergina / Thessaloniki, Macedonia
            (39.1189,  23.0581,   3.0),   # Delphi (Oracle)
            (37.3722,  21.8367,   3.0),   # Ancient Messene, Peloponnese
            (36.4513,  28.2278,   3.0),   # Rhodes old town
            (38.9490,  26.9340,   3.0),   # Pergamon, Turkey
            (38.1820,  27.1660,   3.0),   # Ephesus, Turkey
        ],
    },
    {
        "title": "Silk Road Cities",
        "username": "system",
        "centers": [
            (39.6542,  66.9597,   8.0),   # Samarkand, Uzbekistan
            (39.7747,  64.4286,   8.0),   # Bukhara, Uzbekistan
            (41.5500,  60.6333,   8.0),   # Khiva, Uzbekistan
            (36.2021,  37.1343,   8.0),   # Aleppo, Syria
            (34.9417,  69.2000,   8.0),   # Kabul (historic crossroads)
            (37.5490,  47.0621,   8.0),   # Tabriz, Iran (bazaar)
            (32.6546,  51.6680,   8.0),   # Isfahan, Iran
            (36.8161,  116.9930, 12.0),   # Dunhuang / Mogao Caves, China
        ],
    },

    # ── Polar & Sub-Polar ─────────────────────────────────────────────────────
    {
        "title": "Arctic: Svalbard & Greenland",
        "username": "system",
        "centers": [
            (78.2232,  15.6267,  20.0),   # Longyearbyen, Svalbard
            (77.0000,  15.0000,  60.0),   # Svalbard interior / glaciers
            (64.1814, -51.6941,  20.0),   # Nuuk, Greenland
            (70.0000, -25.0000,  80.0),   # East Greenland fjords
            (69.6492,  18.9553,  15.0),   # Tromsø, Norway (Northern Lights hub)
            (68.3500,  27.0200,  20.0),   # Saariselkä, Finnish Lapland
        ],
    },
    {
        "title": "Antarctica: Research Stations & Ice Shelves",
        "username": "system",
        "centers": [
            (-77.8500, 166.6833,  30.0),  # McMurdo / Scott Base, Ross Island
            (-90.0000,   0.0000,  50.0),  # South Pole (Amundsen-Scott)
            (-75.1000,  -0.0700,  40.0),  # Weddell Sea / Halley Station area
            (-62.0000, -58.3000,  20.0),  # King George Island (many stations)
            (-68.5766,  77.9674,  25.0),  # Mawson Station, East Antarctica
        ],
    },

    # ── Specific Country Deep-Dives ───────────────────────────────────────────
    {
        "title": "Morocco: Imperial Cities & Desert",
        "username": "system",
        "centers": [
            (34.0209,  -5.0089,   8.0),   # Fes medina
            (31.6295,  -7.9811,   8.0),   # Marrakech medina
            (33.9897,  -6.8550,   8.0),   # Rabat historic centre
            (35.5897,  -5.3537,   8.0),   # Chefchaouen (blue city)
            (31.0500,  -4.0000,  50.0),   # Draa Valley / Sahara dunes
            (30.9335,  -6.9370,  10.0),   # Ouarzazate / Aït Benhaddou
        ],
    },
    {
        "title": "Iran: Ancient Persia",
        "username": "system",
        "centers": [
            (29.9350,  52.8917,   8.0),   # Persepolis, Shiraz
            (32.6546,  51.6680,   8.0),   # Isfahan (Naqsh-e Jahan square)
            (35.6892,  51.3890,  10.0),   # Tehran historic centre
            (37.5490,  47.0621,   8.0),   # Tabriz bazaar
            (30.2839,  57.0834,   8.0),   # Bam citadel, Kerman
            (36.2970,  59.6057,   8.0),   # Mashhad / Imam Reza shrine
        ],
    },
    {
        "title": "India: Temples & Palaces",
        "username": "system",
        "centers": [
            (27.1751,  78.0421,   3.0),   # Taj Mahal, Agra
            (26.9124,  75.7873,   8.0),   # Jaipur (Amber Fort / Hawa Mahal)
            (11.0168,  76.9558,   8.0),   # Madurai Meenakshi Temple area
            (15.3350,  75.1300,   8.0),   # Hampi ruins, Karnataka
            (25.3176,  83.0062,   8.0),   # Varanasi / Ganges ghats
            (20.0121,  79.0019,   5.0),   # Ajanta & Ellora caves
            (12.9716,  77.5946,   8.0),   # Bangalore / Mysore Palace area
        ],
    },
    {
        "title": "Nepal, Bhutan & Tibetan Plateau",
        "username": "system",
        "centers": [
            (27.7172,  85.3240,   8.0),   # Kathmandu Valley
            (27.9881,  86.9250,  20.0),   # Sagarmatha / Everest base camp
            (27.4712,  89.6339,   8.0),   # Thimphu, Bhutan
            (27.5014,  90.4336,   5.0),   # Tiger's Nest (Paro Taktsang)
            (29.6520,  91.1721,  12.0),   # Lhasa, Tibet (Potala Palace)
            (28.2090,  83.9855,  15.0),   # Pokhara / Annapurna gateway
        ],
    },
    {
        "title": "Central Asia: Steppes & Mountains",
        "username": "system",
        "centers": [
            (42.8700,  74.5900,  10.0),   # Bishkek, Kyrgyzstan
            (42.1500,  77.5000,  30.0),   # Song-Kol Lake, Kyrgyzstan
            (43.2220,  76.8512,  10.0),   # Almaty, Kazakhstan
            (51.1694,  71.4491,  10.0),   # Nur-Sultan (Astana), Kazakhstan
            (37.9601,  58.3261,  10.0),   # Merv / Mary, Turkmenistan
            (38.9697,  59.5563,   5.0),   # Gonur-depe (ancient Margiana)
        ],
    },
    {
        "title": "Caucasus: Georgia, Armenia & Azerbaijan",
        "username": "system",
        "centers": [
            (41.6938,  44.8015,   8.0),   # Tbilisi old town, Georgia
            (42.2679,  42.7181,   5.0),   # Kutaisi / Gelati monastery
            (40.1431,  44.5152,   8.0),   # Yerevan, Armenia
            (39.8205,  44.9120,   5.0),   # Lake Sevan, Armenia
            (40.4093,  49.8671,   8.0),   # Baku old city, Azerbaijan
            (41.3200,  48.5100,  15.0),   # Greater Caucasus mountain passes
        ],
    },

    # ── European Specialty ────────────────────────────────────────────────────
    {
        "title": "Tuscany & Umbria, Italy",
        "username": "system",
        "centers": [
            (43.7696,  11.2558,   8.0),   # Florence (Firenze)
            (43.3186,  11.3307,  10.0),   # Siena & San Gimignano
            (43.4623,  11.8796,   5.0),   # Arezzo / Cortona
            (42.7197,  12.1131,   5.0),   # Orvieto
            (43.1107,  12.3908,   8.0),   # Perugia / Assisi
            (44.8015,  10.3279,   5.0),   # Parma / Po Valley
            (43.7228,  10.4017,   5.0),   # Pisa (Leaning Tower)
        ],
    },
    {
        "title": "Scottish Highlands & Islands",
        "username": "system",
        "centers": [
            (57.1497,  -2.0943,   8.0),   # Aberdeen / Deeside
            (57.4778,  -4.2247,   8.0),   # Inverness / Loch Ness
            (56.8162,  -5.1050,  20.0),   # Ben Nevis / Fort William
            (57.0488,  -5.7094,  15.0),   # Isle of Skye
            (56.3360,  -6.2611,  20.0),   # Mull & Iona islands
            (60.1542,  -1.1450,  15.0),   # Shetland Islands
            (58.9989,  -2.9609,  12.0),   # Orkney Islands (Skara Brae)
        ],
    },
    {
        "title": "Dalmatian Coast & Adriatic",
        "username": "system",
        "centers": [
            (43.5081,  16.4402,   5.0),   # Split & Diocletian's Palace
            (42.6507,  18.0944,   5.0),   # Dubrovnik old town
            (43.8480,  15.8625,   5.0),   # Šibenik & Krka waterfalls
            (45.3271,  14.4422,  10.0),   # Plitvice Lakes NP
            (44.8667,  13.8500,  10.0),   # Istria peninsula (Pula)
            (42.4411,  19.2636,   8.0),   # Kotor, Montenegro
            (41.3275,  19.8189,   8.0),   # Tirana & Albanian Riviera
        ],
    },
    {
        "title": "Iberian Peninsula: Andalusia & Portugal",
        "username": "system",
        "centers": [
            (37.1761,  -3.5877,   5.0),   # Alhambra, Granada
            (37.3891,  -5.9845,   8.0),   # Seville (Sevilla)
            (37.8882,  -4.7794,   8.0),   # Córdoba Mosque-Cathedral
            (36.5271,  -6.2886,   8.0),   # Cádiz coast
            (38.7223,  -9.1393,   8.0),   # Lisbon (Lisboa)
            (41.1579,  -8.6291,   8.0),   # Porto & Douro Valley
            (37.0193,  -7.9304,  15.0),   # Algarve coastline
        ],
    },
    {
        "title": "Cappadocia & Turkish Heartland",
        "username": "system",
        "centers": [
            (38.6431,  34.8297,  10.0),   # Göreme / fairy chimneys, Cappadocia
            (37.9717,  29.0406,   8.0),   # Pamukkale travertines
            (39.9334,  32.8597,   8.0),   # Ankara old citadel
            (38.0740,  34.8287,   5.0),   # Derinkuyu underground city
            (36.2819,  36.1609,   5.0),   # Antakya (ancient Antioch)
            (37.0662,  37.3833,   8.0),   # Gaziantep (mosaics & cuisine)
        ],
    },
    {
        "title": "Holy Land: Jerusalem & Beyond",
        "username": "system",
        "centers": [
            (31.7683,  35.2137,   5.0),   # Jerusalem Old City
            (31.9000,  35.2000,  10.0),   # Bethlehem / Dead Sea area
            (32.8156,  35.1028,   5.0),   # Nazareth / Sea of Galilee
            (30.3285,  35.4444,   5.0),   # Petra, Jordan
            (29.9259,  35.0266,  10.0),   # Wadi Rum, Jordan
            (31.5000,  34.4667,   8.0),   # Gaza coastal strip area
            (33.5138,  36.2765,   8.0),   # Damascus (oldest city)
        ],
    },

    # ── Americas (New Themes) ─────────────────────────────────────────────────
    {
        "title": "Patagonia: Chile & Argentina",
        "username": "system",
        "centers": [
            (-51.6230, -72.8152,  25.0),  # Torres del Paine NP, Chile
            (-49.3311, -72.8868,  20.0),  # Los Glaciares NP (Perito Moreno)
            (-54.8019, -68.3030,  15.0),  # Ushuaia, Tierra del Fuego
            (-45.8640, -67.5000,  30.0),  # Valdés Peninsula (whales)
            (-39.0000, -71.5000,  30.0),  # Lake District, Bariloche
            (-42.4830, -73.7640,  15.0),  # Chiloé Island, Chile
        ],
    },
    {
        "title": "Canadian Rockies & Pacific Northwest",
        "username": "system",
        "centers": [
            (51.4968, -115.9281,  20.0),  # Banff NP, Alberta
            (52.1166, -117.5543,  20.0),  # Jasper NP, Alberta
            (51.3265, -116.1773,  15.0),  # Yoho NP / Lake Louise area
            (49.1666, -121.9453,  20.0),  # Manning Park, BC
            (48.7596, -121.8000,  20.0),  # North Cascades NP, Washington
            (47.8601, -123.9344,  20.0),  # Olympic Peninsula rainforest
            (49.2827, -123.1207,   8.0),  # Vancouver, BC
        ],
    },
    {
        "title": "US East Coast: Historic Cities",
        "username": "system",
        "centers": [
            (42.3601,  -71.0589,   8.0),  # Boston
            (40.7128,  -74.0060,   8.0),  # New York City
            (39.9526,  -75.1652,   8.0),  # Philadelphia
            (38.9072,  -77.0369,   8.0),  # Washington D.C.
            (39.2904,  -76.6122,   8.0),  # Baltimore
            (32.7765,  -79.9311,   8.0),  # Charleston, SC
            (30.3322,  -81.6557,   8.0),  # St. Augustine (oldest US city)
        ],
    },
    {
        "title": "Mesoamerican Pyramids & Ruins",
        "username": "system",
        "centers": [
            (20.6843,  -88.5678,   5.0),  # Chichén Itzá, Mexico
            (19.6925,  -98.8438,   5.0),  # Teotihuacán (Pyramid of the Sun)
            (17.0519,  -92.0770,   5.0),  # Palenque, Chiapas
            (14.8360,  -91.5182,   5.0),  # Quetzaltenango / Tikal area, Guatemala
            (17.2313,  -89.6230,   5.0),  # Tikal NP, Guatemala
            (15.7217,  -88.9994,   5.0),  # Copán ruins, Honduras
            (13.6994,  -89.1914,   8.0),  # San Salvador / Joya de Cerén
        ],
    },
    {
        "title": "Amazon & Andean Frontiers",
        "username": "system",
        "centers": [
            (-3.1190,  -60.0217,  20.0),  # Manaus, Brazil (Meeting of Waters)
            (-12.0464,  -77.0428,  10.0), # Lima, Peru (coastal desert city)
            (-13.5320,  -71.9675,  15.0), # Sacred Valley, Peru
            (-16.3989,  -71.5350,  10.0), # Arequipa, Peru (El Misti)
            (-11.0000,  -74.0000,  40.0), # Peruvian Amazon (Manu NP)
            ( -4.0000,  -63.0000,  40.0), # Brazilian Amazon confluence zone
        ],
    },

    # ── African Specialty ─────────────────────────────────────────────────────
    {
        "title": "East Africa: Swahili Coast & Zanzibar",
        "username": "system",
        "centers": [
            (-6.1622,  39.1929,  10.0),   # Zanzibar Stone Town
            (-6.8000,  39.2800,  15.0),   # Zanzibar island beaches
            (-4.0435,  39.6682,   8.0),   # Mombasa old town, Kenya
            (-8.9009,  13.1841,   8.0),   # Luanda / Benguela coast, Angola
            (-15.4166,  35.3333,  10.0),  # Lake Malawi shore
            (-10.6869,  40.4925,  10.0),  # Mozambique Island (UNESCO)
        ],
    },
    {
        "title": "West Africa: Empires & Coastlines",
        "username": "system",
        "centers": [
            (16.7736,  -3.0076,   8.0),   # Timbuktu, Mali
            (12.3564,  -1.5353,   8.0),   # Ouagadougou / Bobo-Dioulasso area
            ( 5.5502,  -0.2174,   8.0),   # Accra, Ghana
            ( 6.3654,   2.4183,   8.0),   # Cotonou / Benin coast
            ( 7.4898,   3.9014,   8.0),   # Ibadan, Nigeria
            ( 4.8594,   7.0102,   8.0),   # Niger Delta, Nigeria
            (14.6937, -17.4441,   8.0),   # Dakar / Gorée Island, Senegal
        ],
    },
    {
        "title": "Ethiopian Highlands & Horn of Africa",
        "username": "system",
        "centers": [
            ( 9.0248,  38.7469,   8.0),   # Addis Ababa
            (12.5350,  37.4654,   8.0),   # Gondar / Lalibela area
            (11.9980,  39.0367,   5.0),   # Lalibela rock churches
            (13.4952,  39.4768,   8.0),   # Aksum (ancient obelisks)
            (11.8020,  42.5583,  10.0),   # Djibouti / Gulf of Aden
            ( 2.0469,  45.3182,   8.0),   # Mogadishu (historic port)
        ],
    },

    # ── Oceania & Pacific ─────────────────────────────────────────────────────
    {
        "title": "Polynesia: Remote Islands",
        "username": "system",
        "centers": [
            (-27.1127, -109.3497, 15.0),  # Easter Island (Rapa Nui)
            (-17.5334, -149.5667, 15.0),  # Tahiti, French Polynesia
            (-13.9500, -171.9500, 15.0),  # Samoa
            (-21.1333, -175.2000, 15.0),  # Tonga
            (-19.0544, -169.9156, 10.0),  # Niue
            (-8.5167, 179.2167,   10.0),  # Tuvalu (low-lying atoll)
        ],
    },
    {
        "title": "Micronesia & Western Pacific",
        "username": "system",
        "centers": [
            ( 7.3697,  134.4706,  20.0),  # Palau (Rock Islands)
            ( 7.0897,  171.3810,  15.0),  # Marshall Islands (Majuro)
            ( 6.9147,  158.1611,  10.0),  # Pohnpei / Nan Madol ruins, Micronesia
            (13.4443,  144.7937,  10.0),  # Guam (Chamorro heritage)
            (15.1770,  145.7530,  10.0),  # Saipan, CNMI
            ( 0.5281,  166.9317,  10.0),  # Nauru (smallest republic)
        ],
    },

    # ── Thematic Specialty ────────────────────────────────────────────────────
    {
        "title": "Formula 1 Grand Prix Circuits",
        "username": "system",
        "centers": [
            (43.7347,   7.4205,   3.0),   # Monaco circuit
            (51.5133,  -1.0160,   3.0),   # Silverstone, UK
            (43.9657,   4.8043,   3.0),   # Circuit Paul Ricard, France
            (45.6156,   9.2811,   3.0),   # Monza, Italy
            (50.4372,   5.9711,   3.0),   # Spa-Francorchamps, Belgium
            (25.4849,  55.3958,   3.0),   # Yas Marina, Abu Dhabi
            (30.1327,  31.2044,   3.0),   # Cairo surrounds (old GP route)
            (-25.4905, -49.0940,  3.0),   # Curitiba / Interlagos area, Brazil
        ],
    },
    {
        "title": "Olympic Host Cities: Summer Games",
        "username": "system",
        "centers": [
            (37.9715,  23.7267,   8.0),   # Athens 1896 & 2004
            (48.8566,   2.3522,   8.0),   # Paris 1900 & 1924 & 2024
            (51.5074,  -0.1278,   8.0),   # London 1908, 1948, 2012
            (35.6762, 139.6503,   8.0),   # Tokyo 1964 & 2020
            (55.7558,  37.6173,   8.0),   # Moscow 1980
            (34.0522, -118.2437,  8.0),   # Los Angeles 1932 & 1984
            (-22.9068, -43.1729,  8.0),   # Rio de Janeiro 2016
            (39.9042, 116.4074,   8.0),   # Beijing 2008
        ],
    },
    {
        "title": "Wine Regions of the World",
        "username": "system",
        "centers": [
            (47.0500,   4.8500,  20.0),   # Burgundy (Bourgogne), France
            (44.8378,  -0.5792,  15.0),   # Bordeaux, France
            (41.1579,  -8.6291,  20.0),   # Douro Valley, Portugal
            (37.8882,  -4.7794,  20.0),   # Jerez / Sherry region, Spain
            (43.5500,  11.0500,  20.0),   # Chianti, Tuscany
            (-33.6600,  19.0300,  20.0),  # Stellenbosch / Cape Winelands, S. Africa
            (-33.8688, 151.2093,  15.0),  # Hunter Valley / Barossa area
            (-41.5000,  174.0000, 15.0),  # Marlborough, New Zealand
        ],
    },
    {
        "title": "World's Greatest Waterfalls",
        "username": "system",
        "centers": [
            (-17.9243,  25.8572,  10.0),  # Victoria Falls, Zambia/Zimbabwe
            (-27.1667, -109.3500, 10.0),  # Angel Falls area, Venezuela
            (-25.6947,  -54.4365, 10.0),  # Iguazú Falls, Argentina/Brazil
            (43.0799,  -79.0747,   5.0),  # Niagara Falls, Canada/USA
            (47.5722,   8.0729,   5.0),   # Rhine Falls, Switzerland
            (60.4800,   8.8000,  10.0),   # Vøringsfossen, Norway
            (-43.7350, 170.1097,  10.0),  # Sutherland Falls, New Zealand
        ],
    },
    {
        "title": "Cave Systems & Underground Wonders",
        "username": "system",
        "centers": [
            (17.2694,  104.1783,   8.0),  # Phong Nha caves, Vietnam
            (37.1738, -104.1100,  10.0),  # Carlsbad Caverns NP, USA
            (43.9472,  15.3614,   5.0),   # Postojna Cave, Slovenia
            (-25.8573, -48.6405,  10.0),  # Petar caves, Brazil
            (37.9644,  21.9760,   5.0),   # Cave of Lakes (Kastria), Greece
            (53.2100,  -9.0450,   5.0),   # Aillwee / Doolin caves, Ireland (Burren)
            (-26.6197,  29.9842,   5.0),  # Sudwala Caves, South Africa
        ],
    },
    {
        "title": "UNESCO: Industrial & Engineering Heritage",
        "username": "system",
        "centers": [
            (52.6816,  -1.5083,   5.0),   # Ironbridge Gorge, England (birthplace of industry)
            (51.5833,   7.2500,   8.0),   # Ruhr Valley, Germany (Zollverein)
            (47.0500,   8.3093,   8.0),   # Swiss watch-making towns (Jura)
            (37.5985, 126.9783,   5.0),   # Seoul's Changdeokgung / Hwaseong Fortress
            (55.8642,  -4.2518,   5.0),   # Glasgow (Clyde shipyards)
            (-27.5969, -48.5495,  8.0),   # Laguna / Florianópolis industrial heritage, Brazil
            (45.4654,   9.1859,   5.0),   # Milan's Navigli canal system
        ],
    },
    {
        "title": "Great Lakes of the World",
        "username": "system",
        "centers": [
            (43.7000,  -78.0000,  50.0),  # Lake Ontario
            (44.5000,  -76.5000,  60.0),  # Lake Ontario / Kingston area
            (-3.5000,  29.5000,  60.0),   # Lake Tanganyika (Tanzania/DRC)
            (-12.0000,  34.5000,  40.0),  # Lake Malawi (Lake Nyasa)
            ( 1.0000,  33.0000,  50.0),   # Lake Victoria, East Africa
            (42.1700, -86.6000,  30.0),   # Lake Michigan, Indiana Dunes area
            (61.9500,  30.0000,  30.0),   # Lake Ladoga, Russia
        ],
    },
    {
        "title": "Glaciers & Ice Fields",
        "username": "system",
        "centers": [
            (46.5522,   8.0000,  15.0),   # Aletsch Glacier, Switzerland
            (64.1040, -16.9692,  20.0),   # Vatnajökull ice cap, Iceland
            (-49.3311, -72.8868, 20.0),   # Perito Moreno Glacier, Argentina
            (58.5000, -135.5000, 30.0),   # Glacier Bay NP, Alaska
            (60.4700, -142.9000, 40.0),   # Wrangell-St. Elias / Hubbard Glacier
            (77.0000,  15.0000,  60.0),   # Svalbard ice fields
            (-79.0000,  -85.0000,100.0),  # Antarctic Peninsula glaciers
        ],
    },
    {
        "title": "Sacred Mountains of the World",
        "username": "system",
        "centers": [
            (35.3606, 138.7274,   8.0),   # Mount Fuji, Japan
            (27.9881,  86.9250,  15.0),   # Sagarmatha / Everest, Nepal
            (29.9792,  31.1342,   5.0),   # Gebel el-Mokattam (Cairo sacred hill)
            (-3.0674,  37.3556,  10.0),   # Mount Kilimanjaro, Tanzania
            (37.6387,  22.7276,   5.0),   # Mount Olympus, Greece
            (26.4316,  50.7747,   5.0),   # Jabal al-Nour (Cave of Hira), Mecca area
            (43.3615,  42.4384,  15.0),   # Mount Elbrus, Caucasus
            (47.3667,   0.5000,  10.0),   # Mont Saint-Michel (tidal island), France
        ],
    },
    {
        "title": "Lighthouse Coasts",
        "username": "system",
        "centers": [
            (48.4473,  -4.9650,   8.0),   # Finistère / Pointe du Raz, Brittany
            (58.0000,  -6.5000,  15.0),   # Outer Hebrides, Scotland
            (51.8853,  -8.4945,   8.0),   # Old Head of Kinsale, Ireland
            (55.0000,  14.7500,   8.0),   # Bornholm, Denmark (Baltic)
            (39.4667,  -9.3833,   8.0),   # Cape Roca & Sintra coast, Portugal
            (34.0500,  -6.8300,   8.0),   # Cape Spartel, Morocco (NW tip of Africa)
            (40.9500,  29.1500,   5.0),   # Rumeli & Anadolu lighthouses, Istanbul
        ],
    },
    {
        "title": "Cold War Landmarks & Divided Cities",
        "username": "system",
        "centers": [
            (52.5163,  13.3777,   5.0),   # Berlin Wall route, Germany
            (50.0755,  14.4378,   5.0),   # Prague (1968 Spring)
            (38.9350, 126.9750,  10.0),   # Pyongyang, North Korea (rare zone)
            (37.9525, 126.8800,  10.0),   # DMZ / Panmunjom, Korea
            (35.1300,  33.3300,  10.0),   # Nicosia (divided city), Cyprus
            (55.7558,  37.6173,   8.0),   # Moscow (Cold War HQ)
            (28.6139,  77.2090,   8.0),   # New Delhi (Non-Aligned movement HQ)
        ],
    },
]


# ── Output helpers ────────────────────────────────────────────────────────────

def build_all() -> list:
    """Return a list of dicts ready for geo_preset_create()."""
    result = []
    for p in PRESETS:
        result.append({
            "title":    p["title"],
            "username": p.get("username", "system"),
            "polygons": build_preset_polygons(p["centers"]),
        })
    return result


def print_summary():
    print(f"{'#':<4} {'Title':<50} {'Circles':>7}")
    print("─" * 65)
    for i, p in enumerate(PRESETS, 1):
        print(f"{i:<4} {p['title']:<50} {len(p['centers']):>7}")
    print(f"\nTotal presets: {len(PRESETS)}")


import functions as f

print_summary()
for preset in build_all():
    f.geo_preset_create(preset['title'], preset['username'], preset['polygons'])